In [1]:
import numpy as np
import pyvista as pv

## Loading the mesh

In [2]:
mesh = np.load("cube40_colu_grains8_gsize20.npz")

In [3]:
mesh

NpzFile 'cube40_colu_grains8_gsize20.npz' with keys: knt, ijk

In [4]:
mesh["knt"]

array([[ 20.        ,  20.        , -20.        ],
       [ 20.        ,  20.        ,  20.        ],
       [ 20.        , -20.        ,  20.        ],
       ...,
       [-17.30903499,  -6.76547126,   3.95237661],
       [ -6.16835548, -17.40712315,  -0.63921633],
       [ -6.19750551, -17.4510842 ,  -8.38963781]], shape=(11333, 3))

In [5]:
mesh["ijk"]

array([[11215,  5888,  8325, 11310,     1],
       [11198,  5878,  5860, 11291,     1],
       [ 8294,  8304, 11287,  5912,     1],
       ...,
       [ 3531,  5812,  3552,  3432,     9],
       [ 8186,  3619,  3633,  3541,     9],
       [ 8184,  8219,  8220,  3644,     9]], shape=(51643, 5))

## Creating `pv.UnstructuredGrid`

In [6]:
cells_2D = np.full_like(mesh["ijk"], 4, dtype=np.int32)

In [7]:
cells_2D.shape

(51643, 5)

In [8]:
cells_2D[:, 1:5] = mesh["ijk"][:, 0:4]

In [9]:
cells = cells_2D.flatten(order="C")

In [10]:
cells

array([    4, 11215,  5888, ...,  8219,  8220,  3644],
      shape=(258215,), dtype=int32)

In [11]:
field = pv.UnstructuredGrid(cells, [pv.CellType.TETRA] * mesh["ijk"].shape[0], mesh["knt"])

In [12]:
field

UnstructuredGrid (0x7fe2b5d74b20)
  N Cells:    51643
  N Points:   11333
  X Bounds:   -2.000e+01, 2.000e+01
  Y Bounds:   -2.000e+01, 2.000e+01
  Z Bounds:   -2.000e+01, 2.000e+01
  N Arrays:   0

In [13]:
field.cell_data.set_array(data=mesh["ijk"][:, -1], name="subregions")

In [14]:
field

UnstructuredGrid (0x7fe2b5d74b20)
  N Cells:    51643
  N Points:   11333
  X Bounds:   -2.000e+01, 2.000e+01
  Y Bounds:   -2.000e+01, 2.000e+01
  Z Bounds:   -2.000e+01, 2.000e+01
  N Arrays:   1

## Plotting the mesh

In [15]:
pl = pv.Plotter()
pl.add_mesh_clip_plane(
    field,
    crinkle=True,
    scalars="subregions",
    show_edges=True,
    edge_opacity=0.6,
    cmap="tab20",
);
pl.show()

Widget(value='<iframe src="http://localhost:42233/index.html?ui=P_0x7fe2ad9706e0_0&reconnect=auto" class="pyvi…

In [16]:
pl_2 = pv.Plotter()
pl_2.add_mesh_threshold(
    field,
    scalars="subregions",
    method="lower",
    show_edges=True,
    edge_opacity=0.6,
    cmap="tab20",
);
pl_2.show()

Widget(value='<iframe src="http://localhost:42233/index.html?ui=P_0x7fe28374e990_1&reconnect=auto" class="pyvi…

## Adding vector field to the mesh

In [17]:
field.points # point coordinates

pyvista_ndarray([[ 20.        ,  20.        , -20.        ],
                 [ 20.        ,  20.        ,  20.        ],
                 [ 20.        , -20.        ,  20.        ],
                 ...,
                 [-17.30903499,  -6.76547126,   3.95237661],
                 [ -6.16835548, -17.40712315,  -0.63921633],
                 [ -6.19750551, -17.4510842 ,  -8.38963781]],
                shape=(11333, 3))

In [18]:
from numpy.typing import NDArray, ArrayLike

In [19]:
def vortex_xy(
    points: NDArray,
    centre: ArrayLike,
    core_radius: float,
    vorticity: int,
    polarity_z: int,
) -> NDArray:
    value = np.tile(np.array([0, 0, polarity_z], dtype=float), (points.shape[0], 1))
    centre = np.array(centre, dtype=float)
    move_points = points - centre
    point_location = np.linalg.norm(move_points[:, 0:2], axis=-1)
    points_to_update = point_location > core_radius
    value[points_to_update, 0] = -vorticity * move_points[points_to_update, 1]
    value[points_to_update, 1] = vorticity * move_points[points_to_update, 0]
    value[points_to_update, 2] = np.zeros_like(value[points_to_update, 2])
    return value / np.linalg.norm(value, axis=-1, keepdims=True)

In [27]:
m_value = vortex_xy(
    points=field.points,
    centre=[-10, 5, 0],
    core_radius=3.,
    vorticity=-1,
    polarity_z=1,
)
m_value

array([[ 0.4472136 , -0.89442719,  0.        ],
       [ 0.4472136 , -0.89442719,  0.        ],
       [-0.6401844 , -0.76822128,  0.        ],
       ...,
       [-0.84943537,  0.52769266,  0.        ],
       [-0.98569233, -0.16855455,  0.        ],
       [-0.98595862, -0.16698981,  0.        ]], shape=(11333, 3))

In [28]:
field.point_data["m"] = m_value

In [29]:
field.point_data["m_x"] = field["m"][:, 0]
field.point_data["m_y"] = field["m"][:, 1]
field.point_data["m_z"] = field["m"][:, 2]

In [30]:
field

UnstructuredGrid (0x7fe2b5d74b20)
  N Cells:    51643
  N Points:   11333
  X Bounds:   -2.000e+01, 2.000e+01
  Y Bounds:   -2.000e+01, 2.000e+01
  Z Bounds:   -2.000e+01, 2.000e+01
  N Arrays:   5

## Plotting vector field

In [31]:
pl_3 = pv.Plotter()

In [33]:
pl_3.add_mesh(
    mesh=field.glyph(orient="m", scale=False),
    scalars="m_x",
    cmap="RdBu",
    show_scalar_bar=True,
);
pl_3.add_mesh(
    mesh=field.contour(isosurfaces=[.99, .99], scalars="m_z"),
    color="grey",
    opacity=0.4,
);

In [34]:
pl_3.show()

Widget(value='<iframe src="http://localhost:42233/index.html?ui=P_0x7fe280567110_3&reconnect=auto" class="pyvi…